In [ ]:
library(Seurat)
library(dplyr)
library(data.table)
library(ggplot2)
source("~/Projects/heads/clustering.r")

In [ ]:
data_dir = "/gpfs/gibbs/pi/braun/zy325"

In [ ]:
obj = readRDS("lymphoid_addCgenes/scrcc_lymphoid_c4.rds")

In [ ]:
obj = clustering(obj,keep_c_genes = F,
                plot_QC_metrics = FALSE,
                group.by.vars = "batch_lab",
                harmony_theta=3,dims = 1:20)

In [ ]:
#saveRDS(obj,file="scrcc_lymphoid_c4_clustered.rds")

In [ ]:
obj = FindClusters(obj,resolution = 1.5)

In [ ]:
options(repr.plot.width=8,repr.plot.height=10)
VlnPlot(obj,features = c(
    "CD3D","CD3E","CD3G","TRAC","TRBC1","TRBC2","TRDC",
    "CD8A","CD8B","CD4",
    "NCAM1","FCGR3A",
    "IL7R","TBX21","IFNG",
    "GATA3","PTGDR2","IL1RL1",
    "KIT","RORC","IL23R"
   ),pt.size = 0,stack = TRUE,flip = TRUE)

In [ ]:
options(repr.plot.width=7,repr.plot.height=7)
DimPlot(obj,label=TRUE,reduction = "umap") + NoLegend()

In [ ]:
### TCR mapping ###

tcrs = list.dirs("/gpfs/gibbs/project/braun/zy325/scrcc/raw/tcr_airrflow_output/cellranger",recursive = F)
tcrs = lapply(file.path(tcrs,"outs","filtered_contig_annotations.csv"),function(x){
    cr = fread(x) %>% mutate(
        sample_id2=gsub("^.*cellranger\\/","",gsub("\\/outs.*$","",x)),
        sample_barcode=paste0(sample_id2,'_',barcode))
    return(cr)
}) %>% rbindlist

trbs = tcrs %>% filter(chain == "TRB") 

obj$barcode = gsub("^.*removed_","",obj$name)
obj$sample_barcode = paste0(obj$sample_id2,"_",obj$barcode)
obj$wTCR = obj$sample_barcode %in% tcrs$sample_barcode
obj$wTRB = obj$sample_barcode %in% trbs$sample_barcode

options(repr.plot.width=15,repr.plot.height=7)
DimPlot(obj,group.by = "wTCR") | DimPlot(obj,group.by = "wTRB")

samples_missingtcr = paste0("SCRCC",c("14NORM","15","77","78"))

obj@meta.data %>% 
    filter(!sample_id2 %in% samples_missingtcr) %>%
    group_by(`RNA_snn_res.0.5`) %>%
    summarize(n=n(),
        n_wTCR=sum(wTCR),prop_wTCR=n_wTCR/n,
        n_wTRB=sum(wTRB),prop_wTRB=n_wTRB/n) %>% arrange(desc(prop_wTCR))

In [ ]:
options(repr.plot.width=8.5,repr.plot.height=7)
DimPlot(obj,cells.highlight = obj$name[obj$anno_cd8t == "CD8Tex_NMF3"],sizes.highlight = .1,alpha = 1,pt.size = .1)

In [ ]:
table(obj$`RNA_snn_res.0.5`[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.5`[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.5`[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing=TRUE) #%>% sum

In [ ]:
options(repr.plot.width=16,repr.plot.height=8)

DimPlot(obj,group.by = "batch_lab") | DimPlot(obj,group.by = "batch_seq_rna") #+ NoLegend()

In [ ]:
#obj$lineage3[obj$`RNA_snn_res.0.5` %in% c(0,3,6)] = "ILC2T"

#saveRDS(subset(obj,lineage3=="ILC2T"),file = "scrcc_lymphoid_c4_c0c3c6.rds")

In [ ]:
options(repr.plot.width=15,repr.plot.height=10)
VlnPlot(obj,features = c(
    "CD3D","CD3E","CD3G",
    "TRBC1","TRBC2","TRAC",
    "CD8A","CD8B","CD4","CD69","FOXP3",
    "CD79A","CD79B","MS4A1","MZB1","JCHAIN",
    "NCAM1","NCR1","FCGR3A",
    "MKI67","TOP2A",
    "SPP1","VEGFA",
    "CD68","S100A9","FCN1","C1QC",
    "HBB","PPBP","EPCAM","ALDOB","PECAM1","COL1A1","PTPRC"),pt.size = 0,stack = TRUE,flip = TRUE)

In [ ]:
options(repr.plot.width=15,repr.plot.height=5)
obj = NormalizeData(obj,assay = "AIR")
FeaturePlot(obj,features = c("CD3D","CD3G","TRAC"),ncol = 3)